# 04 — Projection des taux de possession (Source A, 2035/2050)

Étape 1 du bloc C : valider ou invalider la méthode de projection des taux de possession
avant tout calcul de demande. Consommations unitaires gelées (décision prise), ménages
Source A venant du bloc B (`03_split_abc_projete.ipynb`) ; il reste à projeter les taux de
possession des appareils utilisés dans `exctraction of data/source_A_grid_consumption.ipynb`.

**Aucune demande en GWh n'est calculée ici.**

In [1]:
import os
import numpy as np
import pandas as pd

BASE = os.path.dirname(os.getcwd())
CSV_FINAL = os.path.join(BASE, "exctraction of data", "output", "CSV_final.csv")
SPLIT_PATH = os.path.join(os.getcwd(), "output", "split_abc_projete.csv")
OUT_PATH = os.path.join(os.getcwd(), "output", "taux_possession_projetes.csv")

HH_BLOCK = "NÚMERO DE VIVIENDAS POR FUENTE DE ELECTRICIDAD"
CE = "NÚMERO DE HOGARES CON EQUIPAMIENTO DEL HOGAR"
CT = "NÚMERO DE HOGARES CON TECNOLOGÍAS TIC"


## Étape 0 — Inventaire de la donnée disponible

Pour chaque appareil utilisé dans le calcul de demande de `source_A_grid_consumption.ipynb`,
on vérifie dans quelles années `CSV_final.csv` porte une colonne `Tiene`/`No tiene`. Filtre
`MUNICIPIO/TIOC` non vide (élimine les sous-totaux), clé `departamento + provincia + municipio`.

In [2]:
raw = pd.read_csv(CSV_FINAL, encoding="utf-8")
n_raw = len(raw)
mm = raw[raw["MUNICIPIO/TIOC"].notna() & (raw["MUNICIPIO/TIOC"].astype(str).str.strip() != "")].copy()
n_dropped = n_raw - len(mm)
print(f"CSV_final.csv : {n_raw} lignes brutes, {len(mm)} municipalites retenues, {n_dropped} sous-totaux ecartes")
assert len(mm) == 21


CSV_final.csv : 29 lignes brutes, 21 municipalites retenues, 8 sous-totaux ecartes


In [3]:
APPAREILS_DEMANDE = {
    "refrigerador":       (CE, "Refrigerador o congelador"),
    "microondas":         (CE, "Microondas"),
    "calefon":            (CE, "Calefón o termotanque"),
    "aire_acondicionado": (CE, "Aire Acondicionado"),
    "lavadora":           (CE, "Lavadora de ropa"),
    "radio":              (CT, "Radio o equipo de sonido"),
    "television":         (CT, "Televisor"),
    "telefono":           (CT, "Teléfono"),
    "computadora":        (CT, "Computadora o laptop o tablet"),
}
# Testable methodologiquement (historique 2012->2024) mais PAS utilise dans le calcul de demande
# (source_A_grid_consumption.ipynb exclut explicitement Internet fijo : non-reponse trop forte
# et mesure instable dans les petites municipalites de Norte Amazonica).
APPAREILS_HORS_DEMANDE_TESTABLE = {
    "internet": (CT, "Internet fijo en la vivienda"),
}

YEARS = [2001, 2012, 2024]


def years_available(block, item):
    found = []
    for y in YEARS:
        col_t = f"{block} | {y} | {item} | Tiene"
        col_n = f"{block} | {y} | {item} | No tiene"
        if col_t in raw.columns and col_n in raw.columns:
            found.append(y)
    return found


print("Appareils utilises dans le calcul de demande -- annees disponibles dans CSV_final.csv :\n")
for appareil, (block, item) in {**APPAREILS_DEMANDE, **APPAREILS_HORS_DEMANDE_TESTABLE}.items():
    ys = years_available(block, item)
    tag = "" if appareil in APPAREILS_DEMANDE else "  (hors demande, testable seulement)"
    print(f"  {appareil:<20}: {ys}{tag}")


Appareils utilises dans le calcul de demande -- annees disponibles dans CSV_final.csv :

  refrigerador        : [2024]
  microondas          : [2024]
  calefon             : [2024]
  aire_acondicionado  : [2024]
  lavadora            : [2024]
  radio               : [2001, 2012, 2024]
  television          : [2001, 2012, 2024]
  telefono            : [2001, 2012, 2024]
  computadora         : [2012, 2024]
  internet            : [2012, 2024]  (hors demande, testable seulement)


### Appareils utilisés dans le calcul de demande sans colonne `Tiene`/`No tiene`

Trois cas dans `source_A_grid_consumption.ipynb` :

- **Antenne** (proxy Internet/câble, end-use ELECTRICITY) : réutilise la pénétration de la
  **radio**, aucune colonne propre — rien à projeter séparément.
- **Ventilateur** (end-use SPACE_COOLING) : vient de `data/thermal_comfort_lookup.csv`
  (kWh/HH/saison, dérivé du confort thermique), pas d'un ratio Tiene/No tiene du recensement —
  hors du cadre de cette méthode, ne peut pas être projeté ainsi.
- **Mixeur/blender** (end-use ELECTRICITY) : appliqué à 100 % des ménages raccordés dans le
  calcul de demande, sans facteur de pénétration — rien à projeter, aucun taux de possession
  n'entre dans son calcul.

Voiture et moto ont un historique complet (2001/2012/2024) mais ne sont pas utilisés dans le
calcul de demande (mobilité nulle dans le modèle) — ignorés à partir d'ici.

## Étape 1 — Le test qui décide de la méthode

Hypothèse : le taux de possession d'un appareil est fonction du taux d'accès à l'électricité
de la municipalité, et cette relation est stable dans le temps. Cinq appareils sont testables
(historique) : radio, télévision, téléphone, ordinateur, internet.

Taux d'accès recalculé depuis `CSV_final.csv` (même définition que
`03_split_abc_projete.ipynb`) :
- 2012, 2024 : `(Total − No tiene) / Total`

Protocole par appareil testable : ajustement OLS `possession_2012 = a·acces_2012 + b` sur les
21 municipalités, prédiction sur `acces_2024`, comparaison à la possession 2024 observée (MAE
en points de %), contre le gel (`possession_2024_prédite = possession_2012`).

Seuil de décision fixé avant le calcul : `f(accès)` retenue si elle bat le gel sur **≥ 3/5**
appareils **et** son avantage moyen de MAE sur les 5 est **≥ 2 points**.

In [4]:
mm["acc_2012"] = (mm[f"{HH_BLOCK} | 2012 | Total"] - mm[f"{HH_BLOCK} | 2012 | No tiene"]) / mm[f"{HH_BLOCK} | 2012 | Total"]
mm["acc_2024"] = (mm[f"{HH_BLOCK} | 2024 | Total"] - mm[f"{HH_BLOCK} | 2024 | No tiene"]) / mm[f"{HH_BLOCK} | 2024 | Total"]

assert mm[["acc_2012", "acc_2024"]].apply(lambda s: s.between(0, 1)).all().all()
mm[["DEPARTAMENTO", "MUNICIPIO/TIOC", "acc_2012", "acc_2024"]].round(4)


,DEPARTAMENTO,MUNICIPIO/TIOC,acc_2012,acc_2024
1,La Paz,Ixiamas,0.5656,0.6845
3,Beni,Riberalta,0.8692,0.8934
4,Beni,Guayaramerín,0.8962,0.9242
5,Beni,Reyes,0.6779,0.7919
6,Beni,Santa Rosa,0.8426,0.8523
7,Beni,Exaltación,0.7111,0.8124
10,Pando,Cobija,0.9173,0.9746
11,Pando,Porvenir,0.7644,0.9045
12,Pando,Bolpebra,0.4422,0.6845
13,Pando,Bella Flor,0.5561,0.7158


In [5]:
TESTABLE = {**APPAREILS_DEMANDE, **APPAREILS_HORS_DEMANDE_TESTABLE}
TESTABLE = {k: TESTABLE[k] for k in ["radio", "television", "telefono", "computadora", "internet"]}


def posesion(block, item, year):
    tiene = mm[f"{block} | {year} | {item} | Tiene"].astype(float)
    no_tiene = mm[f"{block} | {year} | {item} | No tiene"].astype(float)
    return (tiene / (tiene + no_tiene)).values


acc12 = mm["acc_2012"].values
acc24 = mm["acc_2024"].values

rows_t1 = []
for appareil, (block, item) in TESTABLE.items():
    p12 = posesion(block, item, 2012)
    p24 = posesion(block, item, 2024)

    a, b = np.polyfit(acc12, p12, 1)
    pred12 = a * acc12 + b
    ss_res = np.sum((p12 - pred12) ** 2)
    ss_tot = np.sum((p12 - p12.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot

    pred24_f = np.clip(a * acc24 + b, 0, 1)
    pred24_gel = p12

    mae_f = np.mean(np.abs(pred24_f - p24)) * 100
    mae_gel = np.mean(np.abs(pred24_gel - p24)) * 100
    n_better = int(np.sum(np.abs(pred24_f - p24) < np.abs(pred24_gel - p24)))

    rows_t1.append(dict(appareil=appareil, mae_f=mae_f, mae_gel=mae_gel,
                         ecart=mae_gel - mae_f, r2_2012=r2, n_munis_f_meilleur=n_better,
                         a_2012=a, b_2012=b))

test1 = pd.DataFrame(rows_t1)
test1.round(3)


,appareil,mae_f,mae_gel,ecart,r2_2012,n_munis_f_meilleur,a_2012,b_2012
0,radio,10.952,8.196,-2.756,0.214,9,0.307,0.295
1,television,9.583,7.293,-2.290,0.865,8,0.943,-0.187
2,telefono,38.009,51.309,13.300,0.766,18,0.908,-0.262
3,computadora,4.669,6.528,1.859,0.540,16,0.285,-0.102
4,internet,10.936,12.220,1.284,0.378,17,0.098,-0.043


In [6]:
n_appareils_f_gagne = int((test1["mae_f"] < test1["mae_gel"]).sum())
avg_ecart = test1["ecart"].mean()

print(f"Appareils ou f(acces) bat le gel (MAE globale sur 21 municipalites) : {n_appareils_f_gagne}/5")
print(f"Avantage moyen de MAE (gel - f) sur les 5 appareils : {avg_ecart:+.3f} points de pourcentage")

SEUIL_N, SEUIL_PP = 3, 2.0
test_passe = (n_appareils_f_gagne >= SEUIL_N) and (avg_ecart >= SEUIL_PP)
print(f"\nSeuil : >= {SEUIL_N}/5 ET >= {SEUIL_PP} pp -- {'PASSE' if test_passe else 'ECHOUE'}")
if not test_passe:
    raise SystemExit(
        f"Seuil non franchi (n={n_appareils_f_gagne}/5, ecart={avg_ecart:.3f}pp < {SEUIL_PP}) -- "
        "ARRET. Pas de methode de repli choisie ici."
    )


Appareils ou f(acces) bat le gel (MAE globale sur 21 municipalites) : 3/5
Avantage moyen de MAE (gel - f) sur les 5 appareils : +2.279 points de pourcentage

Seuil : >= 3/5 ET >= 2.0 pp -- PASSE


### Résultat étape 1

Le test **passe**, mais près de la limite fixée à l'avance : 3/5 appareils (téléphone,
ordinateur, internet) où `f(accès)` bat le gel, avantage moyen +2.28 points — au-dessus du
seuil de 2.0 mais avec une faible marge. Radio et télévision sont mieux gelées que projetées
(la relation à l'accès y est faible ou négative en termes de MAE), téléphone en particulier
montre un écart énorme en faveur de `f(accès)` (+13.3 points), tiré par une pénétration
téléphonique qui a explosé entre 2012 et 2024 indépendamment de l'accès électrique — c'est ce
seul appareil qui fait basculer la moyenne au-dessus du seuil. On continue à l'étape 2, mais ce
résultat mérite d'être lu avec cette réserve : la décision repose sur un seuil franchi de
justesse et porté par un seul appareil à fort effet.

## Étape 2 — Robustesse de l'ajustement (sans Cobija)

**Note** : l'hypothèse énoncée dans la consigne ("Cobija est la seule municipalité au-dessus de
90 % d'accès") ne se vérifie pas dans les données recalculées ci-dessous — Guayaramerín et
Porvenir dépassent aussi 90 % en 2024. Cobija reste la valeur la plus extrême (accès le plus
élevé, 97.5 %) et le diagnostic demandé (retirer Cobija spécifiquement) est effectué tel quel ;
il ne couvre donc pas tout le haut de la courbe, seulement son point le plus extrême.

In [7]:
top_acc = mm[["MUNICIPIO/TIOC", "acc_2024"]].sort_values("acc_2024", ascending=False).reset_index(drop=True)
print("Municipalites au-dessus de 90% d'acces en 2024 :")
print(top_acc[top_acc["acc_2024"] > 0.90].to_string(index=False))
print(f"\n(Cobija = {top_acc['acc_2024'].iloc[0]:.4f}, la plus elevee mais pas la seule > 0.90)")


Municipalites au-dessus de 90% d'acces en 2024 :
MUNICIPIO/TIOC  acc_2024
        Cobija  0.974621
  Guayaramerín  0.924249
      Porvenir  0.904484

(Cobija = 0.9746, la plus elevee mais pas la seule > 0.90)


In [8]:
is_cobija = (mm["MUNICIPIO/TIOC"].str.strip() == "Cobija").values

rows_t2 = []
for appareil, (block, item) in TESTABLE.items():
    p12 = posesion(block, item, 2012)

    a_avec, b_avec = np.polyfit(acc12, p12, 1)
    pred1_avec = np.clip(a_avec * 1.0 + b_avec, 0, 1)

    mask = ~is_cobija
    a_sans, b_sans = np.polyfit(acc12[mask], p12[mask], 1)
    pred1_sans = np.clip(a_sans * 1.0 + b_sans, 0, 1)

    rows_t2.append(dict(appareil=appareil, a_avec_Cobija=a_avec, b_avec_Cobija=b_avec,
                         pred_acc1_avec=pred1_avec, a_sans_Cobija=a_sans, b_sans_Cobija=b_sans,
                         pred_acc1_sans=pred1_sans, ecart_pred_acc1=pred1_sans - pred1_avec))

test2 = pd.DataFrame(rows_t2)
test2.round(4)


,appareil,a_avec_Cobija,b_avec_Cobija,pred_acc1_avec,a_sans_Cobija,b_sans_Cobija,pred_acc1_sans,ecart_pred_acc1
0,radio,0.3066,0.2947,0.6013,0.2977,0.2991,0.5968,-0.0045
1,television,0.9431,-0.1871,0.7560,0.8897,-0.1605,0.7292,-0.0268
2,telefono,0.9081,-0.2617,0.6463,0.8211,-0.2184,0.6028,-0.0436
3,computadora,0.2853,-0.1021,0.1832,0.2068,-0.0629,0.1439,-0.0394
4,internet,0.0977,-0.0431,0.0546,0.0568,-0.0227,0.0341,-0.0205


Retirer Cobija déplace la prédiction à `acces = 1` de quelques points de pourcentage pour
tous les appareils testables (de −0.5 pp pour la radio à −4.4 pp pour le téléphone), toujours
dans le même sens (la pente est plus faible sans Cobija). Le haut de la courbe est donc
sensible à ce point unique, mais pas au point de renverser la conclusion de l'étape 1 — c'est un
diagnostic, aucune correction n'est appliquée.

## Étape 3 — Ajustement 2024

Ajustement `possession_2024 = a·acces_2024 + b` sur les 21 municipalités, pour les cinq
appareils sans historique (réfrigérateur, micro-ondes, chauffe-eau, climatisation, machine à
laver) et pour les cinq appareils testables (recalculé sur 2024, puisque c'est cet ajustement
qui sert à la projection en étape 4).

In [9]:
NON_HISTORIQUE = {k: APPAREILS_DEMANDE[k] for k in
                  ["refrigerador", "microondas", "calefon", "aire_acondicionado", "lavadora"]}
ALL_APPAREILS_2024 = {**NON_HISTORIQUE, **TESTABLE}

cobija_acc24 = mm.loc[is_cobija, "acc_2024"].iloc[0]

rows_t3 = []
for appareil, (block, item) in ALL_APPAREILS_2024.items():
    p24 = posesion(block, item, 2024)
    a, b = np.polyfit(acc24, p24, 1)
    pred = a * acc24 + b
    ss_res = np.sum((p24 - pred) ** 2)
    ss_tot = np.sum((p24 - p24.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot
    pred_acc1 = np.clip(a * 1.0 + b, 0, 1)
    obs_cobija = p24[is_cobija][0]
    rows_t3.append(dict(appareil=appareil, a=a, b=b, r2_2024=r2, pred_acc1=pred_acc1,
                         obs_cobija=obs_cobija, testable=appareil in TESTABLE))

coef2024 = pd.DataFrame(rows_t3)
coef2024.round(4)


,appareil,a,b,r2_2024,pred_acc1,obs_cobija,testable
0,refrigerador,1.2259,-0.6378,0.7050,0.5881,0.7562,False
1,microondas,0.1848,-0.1081,0.4925,0.0767,0.1358,False
2,calefon,0.0450,-0.0156,0.2856,0.0294,0.0353,False
3,aire_acondicionado,0.2364,-0.1478,0.4479,0.0887,0.1887,False
4,lavadora,0.8022,-0.4661,0.5360,0.3361,0.5479,False
5,radio,0.0079,0.4377,0.0001,0.4456,0.5721,True
6,television,0.9534,-0.2888,0.7519,0.6646,0.7451,True
7,telefono,0.3699,0.5281,0.6055,0.8981,0.9375,True
8,computadora,0.4933,-0.2370,0.5121,0.2564,0.4310,True
9,internet,0.8452,-0.5044,0.4758,0.3408,0.6280,True


In [10]:
faibles = coef2024.loc[coef2024["r2_2024"] < 0.3, "appareil"].tolist()
print(f"Appareils avec R2 2024 < 0.3 (l'acces n'explique pas l'equipement) : {faibles}")


Appareils avec R2 2024 < 0.3 (l'acces n'explique pas l'equipement) : ['calefon', 'radio']


Radio (R² ≈ 0.0001) et chauffe-eau/calefón (R² ≈ 0.29) sont sous le seuil de 0.3 : pour
ces deux appareils, l'accès à l'électricité n'explique quasiment rien de la variance de
possession en 2024 — la méthode `f(accès)` n'a pas de justification statistique pour eux et sa
projection doit être lue comme peu fiable, même si elle est produite en étape 4 pour rester
cohérente avec les autres appareils.

## Étape 4 — Projection des taux de possession

Pour chaque appareil, municipalité, année (2035, 2050) et trajectoire (`acces_2035`,
`acces_2050`) :

    possession(m,t) = clip( a·acces(m,t) + b , 0, 1 )

`acces(m,t)` vient de `split_abc_projete.csv` (colonne `acc`). Deux variantes : **centrale**
(`f(accès)`) et **basse** (possession gelée à sa valeur 2024 observée). En 2024, les deux
variantes reproduisent le taux observé — contrôle, pas projection.

In [11]:
split_abc = pd.read_csv(SPLIT_PATH)
assert len(split_abc) == 126
acc_long = split_abc[["departamento", "provincia", "municipio", "cluster", "trajectoire", "annee", "acc"]].copy()

APPAREILS_ORDER = list(ALL_APPAREILS_2024.keys())

obs2024_rows = []
for _, row in mm.iterrows():
    for appareil, (block, item) in ALL_APPAREILS_2024.items():
        tiene = float(row[f"{block} | 2024 | {item} | Tiene"])
        no_tiene = float(row[f"{block} | 2024 | {item} | No tiene"])
        obs2024_rows.append(dict(
            departamento=row["DEPARTAMENTO"], provincia=row["PROVINCIA"],
            municipio=row["MUNICIPIO/TIOC"], appareil=appareil,
            possession_2024_obs=tiene / (tiene + no_tiene),
        ))
obs2024 = pd.DataFrame(obs2024_rows)
assert len(obs2024) == 21 * len(APPAREILS_ORDER)

scaffold = acc_long.merge(pd.DataFrame({"appareil": APPAREILS_ORDER}), how="cross")
assert len(scaffold) == 126 * len(APPAREILS_ORDER)

merged = scaffold.merge(obs2024, on=["departamento", "provincia", "municipio", "appareil"],
                         how="left", indicator=True)
assert (merged["_merge"] == "both").all(), merged.loc[merged["_merge"] != "both"]
merged = merged.drop(columns="_merge")

merged = merged.merge(coef2024[["appareil", "a", "b"]], on="appareil", how="left", indicator=True)
assert (merged["_merge"] == "both").all()
merged = merged.drop(columns="_merge")

print(f"scaffold fusionne : {merged.shape} (attendu ({126*len(APPAREILS_ORDER)}, 11))")
merged.head()


scaffold fusionne : (1260, 11) (attendu (1260, 11))


,departamento,provincia,municipio,cluster,trajectoire,annee,acc,appareil,possession_2024_obs,a,b
0,La Paz,Abel Iturralde,Ixiamas,C1,acces_2035,2024,0.684513,refrigerador,0.269506,1.225945,-0.637841
1,La Paz,Abel Iturralde,Ixiamas,C1,acces_2035,2024,0.684513,microondas,0.034774,0.184780,-0.108093
2,La Paz,Abel Iturralde,Ixiamas,C1,acces_2035,2024,0.684513,calefon,0.028599,0.045013,-0.015636
3,La Paz,Abel Iturralde,Ixiamas,C1,acces_2035,2024,0.684513,aire_acondicionado,0.014304,0.236437,-0.147769
4,La Paz,Abel Iturralde,Ixiamas,C1,acces_2035,2024,0.684513,lavadora,0.090052,0.802181,-0.466084


In [12]:
is_2024 = merged["annee"] == 2024
pred_centrale_raw = np.clip(merged["a"] * merged["acc"] + merged["b"], 0, 1)

merged["possession_centrale"] = np.where(is_2024, merged["possession_2024_obs"], pred_centrale_raw)
merged["possession_basse"] = merged["possession_2024_obs"]

out_centrale = merged.copy()
out_centrale["variante"] = "centrale"
out_centrale["possession"] = out_centrale["possession_centrale"]

out_basse = merged.copy()
out_basse["variante"] = "basse"
out_basse["possession"] = out_basse["possession_basse"]

OUT_COLS = ["departamento", "provincia", "municipio", "cluster", "appareil", "variante",
            "trajectoire", "annee", "acc", "possession"]
out = pd.concat([out_centrale[OUT_COLS], out_basse[OUT_COLS]], ignore_index=True)
out = out.rename(columns={"acc": "acces"})

print(f"out shape = {out.shape} (attendu ({126*len(APPAREILS_ORDER)*2}, {len(OUT_COLS)}))")
out.head()


out shape = (2520, 10) (attendu (2520, 10))


,departamento,provincia,municipio,cluster,appareil,variante,trajectoire,annee,acces,possession
0,La Paz,Abel Iturralde,Ixiamas,C1,refrigerador,centrale,acces_2035,2024,0.684513,0.269506
1,La Paz,Abel Iturralde,Ixiamas,C1,microondas,centrale,acces_2035,2024,0.684513,0.034774
2,La Paz,Abel Iturralde,Ixiamas,C1,calefon,centrale,acces_2035,2024,0.684513,0.028599
3,La Paz,Abel Iturralde,Ixiamas,C1,aire_acondicionado,centrale,acces_2035,2024,0.684513,0.014304
4,La Paz,Abel Iturralde,Ixiamas,C1,lavadora,centrale,acces_2035,2024,0.684513,0.090052


### Assertions bloquantes

In [13]:
n_munis = out.groupby(["departamento", "provincia", "municipio"]).ngroups
assert n_munis == 21, n_munis
print(f"1. municipalites distinctes : {n_munis} -- OK")

assert out["possession"].between(0, 1).all()
print("2. tous les taux de possession dans [0,1] -- OK")

assert out.isna().sum().sum() == 0
print("3. aucun NaN dans la sortie -- OK")

print("4. merge avec split_abc_projete.csv : indicator verifie ci-dessus (both partout) -- OK")

basse_2024 = out[(out["variante"] == "basse") & (out["annee"] == 2024)].set_index(
    ["departamento", "provincia", "municipio", "appareil"])["possession"]
for y in [2035, 2050]:
    basse_y = out[(out["variante"] == "basse") & (out["annee"] == y)].set_index(
        ["departamento", "provincia", "municipio", "appareil", "trajectoire"])["possession"]
    ref = basse_2024.reindex(basse_y.index.droplevel("trajectoire")).values
    assert np.allclose(basse_y.values, ref, atol=1e-9), f"variante basse {y} != 2024 observe"
print("5. variante basse 2035/2050 == taux 2024 observe (1e-9) -- OK")

violations = []
for (dep, prov, muni, appareil, traj), sub in out[out["variante"] == "centrale"].groupby(
        ["departamento", "provincia", "municipio", "appareil", "trajectoire"]):
    piv = sub.set_index("annee")["possession"]
    if piv[2050] < piv[2035] - 1e-9:
        violations.append((dep, prov, muni, appareil, traj, piv[2035], piv[2050]))
print(f"6. violations de non-regression (possession(2050) < possession(2035), centrale) : {len(violations)}")
for v in violations:
    print("  ", v)


1. municipalites distinctes : 21 -- OK
2. tous les taux de possession dans [0,1] -- OK
3. aucun NaN dans la sortie -- OK
4. merge avec split_abc_projete.csv : indicator verifie ci-dessus (both partout) -- OK
5. variante basse 2035/2050 == taux 2024 observe (1e-9) -- OK


6. violations de non-regression (possession(2050) < possession(2035), centrale) : 0


In [14]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
out.to_csv(OUT_PATH, index=False)
print(f"ecrit : {OUT_PATH}  ({len(out)} lignes, {len(out.columns)} colonnes)")


ecrit : c:\Valen\Tfe\bolivia-energy-data\projections\output\taux_possession_projetes.csv  (2520 lignes, 10 colonnes)


### Taux de possession régionaux moyens (moyenne non pondérée sur 21 municipalités)

Variante centrale, trajectoire `acces_2035` — ordre de grandeur seulement.

In [15]:
region_avg = out[(out["variante"] == "centrale") & (out["trajectoire"] == "acces_2035")].groupby(
    ["appareil", "annee"])["possession"].mean().unstack("annee")
region_avg.round(3)


annee,2024,2035,2050
appareil,,,
aire_acondicionado,0.032,0.089,0.089
calefon,0.019,0.029,0.029
computadora,0.139,0.256,0.256
internet,0.139,0.341,0.341
lavadora,0.145,0.336,0.336
microondas,0.033,0.077,0.077
radio,0.444,0.446,0.446
refrigerador,0.296,0.588,0.588
telefono,0.810,0.898,0.898


## Conclusion

Le test de l'étape 1 passe, de justesse : 3/5 appareils testables battent le gel, avantage
moyen +2.28 points (seuil 2.0). Ce résultat est porté presque entièrement par le téléphone, dont
la pénétration a explosé entre 2012 et 2024 indépendamment de l'accès électrique. Radio et
télévision sont en réalité mieux servies par le gel. L'étape 2 confirme une sensibilité réelle
mais modérée au point Cobija. En étape 3, deux appareils (radio, calefón) ont un R² 2024 sous
0.3 — leur projection par `f(accès)` n'a pas de fondement statistique solide et doit être
utilisée avec cette réserve explicite dans l'étape suivante (calcul de demande).